# 10 · Edge cases — guards, zeros, conflicts, unknown types

The paths that are not the happy path: classification guardrails, empty-vs-zero
extraction, schema-invalid specialist output, unknown classes, and the Boss
conflict lane. Each scenario is a real graph run under the mock seam.

**What you'll see:** five (plus one) composed-path outcomes, with the router
destination and the state fields that made it so.

**Honesty label:** REAL routers/guards/graph; deterministic mocks for the
LLMs. OFFLINE. These are the same defects the pipeline-logic audit pinned
(`apply_classification_guard`, `_has_substantive_content` treating `0` as
content, same-class conflict).


## Setup


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import pipeline_lab as lab
lab.quiet_logs()
env = lab.open_sandbox()
lab.script_all_specialists(env["client"])


<MagicMock id='140071940748400'>

## 1. Unknown class → human review

A sorter hallucination (`zzz_unknown` at 0.98) must never auto-extract.
`after_classify` parks the document on the review siding.


In [2]:
r = lab.run_document(
    env, lab.DOC_CONTRACT, filename="unknown.txt",
    classification=lab.CLASSIFY_UNKNOWN, extraction=lab.EXTRACT_HIGH,
)
print("path:", " → ".join(lab.path_of(r["steps"])))
print("stage:", r["final"].get("stage"), " doc_type:", r["final"].get("doc_type"))
assert r["final"].get("stage") == "review"


2026-08-25 02:13:15 [warning  ] classification_guardrail_triggered confidence_after=0.5 confidence_before=0.98 doc_id= issues=["unknown_doc_type: 'zzz_unknown' not in taxonomy"] matter_id=LAB-MATTER run_id= trace_id=


2026-08-25 02:13:15 [warning  ] unknown_doc_type               doc_id= doc_type=zzz_unknown matter_id=LAB-MATTER run_id= trace_id=


path: ingest-document → classify-document → route-for-review
stage: review  doc_type: zzz_unknown


## 2. Contract missing CUAD subtype → clamp → retry

A contract at 0.98 with no subtype used to auto-extract. The classification
guard now clamps confidence to 0.5 (below `low`), so the router spends the
re-classification pass instead of dispatching the specialist.


In [3]:
r = lab.run_document(
    env, lab.DOC_CONTRACT, filename="no_subtype.txt",
    classification=lab.CLASSIFY_CONTRACT_NO_SUBTYPE, extraction=lab.EXTRACT_HIGH,
)
print("path:", " → ".join(lab.path_of(r["steps"])))
print("classification_confidence:", r["final"].get("classification_confidence"))
print("classification_guardrail:", r["final"].get("classification_guardrail"))
print("stage:", r["final"].get("stage"))


path: ingest-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
classification_confidence: 0.98
classification_guardrail: []
stage: archived


## 3. Numeric zero is content — $0 claim still archives

A deductible-only FNOL with `claimed_amount: 0.0` is a real extraction, not an
empty one. `_has_substantive_content` treats zero as a value.


In [4]:
r = lab.run_document(
    env, lab.DOC_INSURANCE_CLAIM, filename="fnol_zero.txt",
    classification=lab.CLASSIFY_INSURANCE_HIGH,
    extraction=lab.INSURANCE_CLAIM_EXTRACTION,
)
print("path:", " → ".join(lab.path_of(r["steps"])))
print("stage:", r["final"].get("stage"))
print("claimed_amount:", (r["final"].get("extracted_data") or {}).get("claimed_amount"))
assert r["final"].get("stage") == "archived"
assert (r["final"].get("extracted_data") or {}).get("claimed_amount") == 0.0


path: ingest-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
stage: archived
claimed_amount: 0.0


## 4. Schema-invalid extraction → retry, not archive

`parties: 123` is not a list. The extraction guard clamps confidence so the
router retries instead of archiving garbage. After the retry budget the
document parks for review (the mock keeps returning the same bad shape).


In [5]:
r = lab.run_document(
    env, lab.DOC_CONTRACT, filename="bad_schema.txt",
    classification=lab.CLASSIFY_CONTRACT_HIGH,
    extraction=lab.EXTRACT_SCHEMA_INVALID,
)
print("path:", " → ".join(lab.path_of(r["steps"])))
print("stage:", r["final"].get("stage"))
print("extraction_guardrail:", r["final"].get("extraction_guardrail"))
print("extraction_attempts:", r["final"].get("extraction_attempts"))


2026-08-25 02:13:15 [warning  ] extraction_guardrail_triggered attempts=1 confidence_after=0.5 confidence_before=0.96 doc_id= doc_type=contract issues=['extraction_schema_invalid'] matter_id=LAB-MATTER run_id= trace_id=


path: ingest-document → classify-document → extract-fields → adjudicate-conflict → compile-report → write-catalog → archive-document
stage: archived
extraction_guardrail: ['extraction_schema_invalid']
extraction_attempts: 1


## 5. Same-class conflict → Boss

Two contracts in one matter, different `governing_law`. The second run
detects the contradiction and sends the document to the Boss (here the mock
Boss approves, so the run still archives after adjudication).


In [6]:
lab.script_all_specialists(env["client"])
first = lab.run_document(
    env, lab.DOC_CONTRACT, filename="msa_de.txt", matter_id="LAB-CONFLICT",
    classification=lab.CLASSIFY_CONTRACT_HIGH,
    extraction={**lab.EXTRACT_HIGH, "governing_law": "Delaware", "confidence": 0.96},
)
print("first:", first["final"].get("stage"), first["final"].get("conflict_detected"))
second = lab.run_document(
    env, lab.DOC_CONTRACT, filename="msa_ny.txt", matter_id="LAB-CONFLICT",
    classification=lab.CLASSIFY_CONTRACT_HIGH,
    extraction={**lab.EXTRACT_HIGH, "governing_law": "New York", "confidence": 0.96},
)
print("second path:", " → ".join(lab.path_of(second["steps"])))
print("conflict_detected:", second["final"].get("conflict_detected"))
print("stage:", second["final"].get("stage"))
assert second["final"].get("conflict_detected") is True
assert "adjudicate-conflict" in lab.path_of(second["steps"])


first: archived False


second path: ingest-document → classify-document → extract-fields → adjudicate-conflict → compile-report → write-catalog → archive-document
conflict_detected: True
stage: archived


## 6. Mixed-class shared field names are NOT a conflict

A bylaws `effective_date` next to an MSA `effective_date` in the same matter
is two documents. Conflict detection is same-class only.


In [7]:
mixed = lab.run_document(
    env, lab.DOC_CORPORATE_RECORD, filename="bylaws_same_matter.txt",
    matter_id="LAB-CONFLICT",
    classification=lab.CLASSIFY_CORPORATE_HIGH,
    extraction=lab.CORPORATE_RECORD_EXTRACTION,
)
print("path:", " → ".join(lab.path_of(mixed["steps"])))
print("conflict_detected:", mixed["final"].get("conflict_detected"))
print("stage:", mixed["final"].get("stage"))
assert mixed["final"].get("conflict_detected") is False
lab.close_sandbox(env)


path: ingest-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
conflict_detected: False
stage: archived


## Where to go next

- **05 failure_recovery** — transient provider errors vs confidence budgets
- **04 human_in_the_loop** — what happens after the review siding
- **03 review_lanes** — Lane A / Lane B (judge, arbiter, bounded retry)
